In [1]:
import numpy as np

from phase_II.utils.helpers import visualize_stress, usual_plot, Stress, unpickle_me_this
from phase_I.utils.config_jupyter_notebooks import *
%matplotlib tk

nrt_strain_values = np.loadtxt("../../data/data_txt/num_rel_template_strain_values.txt") * 1e19
nrt_time_values = np.loadtxt("../../data/data_txt/num_rel_template_time_values.txt") - zero_time

Important variables: 
		signal_strip_time, signal_strip_strain 
		signal_strip_strain_tapered
		strain
		time_domain_strip
		N


In [2]:
from phase_II.nifty_re_playground.useful.helpers import get_sample_data

S_mat_inference, t_dual_inference, f_inference = unpickle_me_this("../wigner_result_pipe_2.pickle")
time_tmp, _ = get_sample_data()
t_dual_inference = t_dual_inference + min(time_tmp)  # In my head the grav wave starts at 16.4 not at 1.4

N = len(t_dual_inference)
dt = t_dual_inference[1]-t_dual_inference[0]
time_dom = ift.RGSpace(shape=(N), distances=dt)

In [ ]:
# Is it possible to average out the background?

def generate_white_noise_stress_matrices(number_of_matrices, time_domain, supress_print=False):
    S_mat_collection = []
    for _ in range(number_of_matrices):
            real_space_white_noise = ift.from_random(time_domain)
            stress, _, _ = Stress(real_space_white_noise, supress_print=supress_print)
            S_mat_collection.append(np.array(stress.real))

    return S_mat_collection


In [ ]:
white_noise_ensemble = generate_white_noise_stress_matrices(number_of_matrices=10, time_domain=time_dom, supress_print=True)

In [ ]:
# Generate more
second_white_noise_ensemble = generate_white_noise_stress_matrices(number_of_matrices=30, time_domain=time_dom)

In [ ]:
white_noise_ensemble_prime = np.array(white_noise_ensemble)
second_white_noise_ensemble_prime = np.array(second_white_noise_ensemble)

# pre-allocate
out = np.empty((41, 8191, 8191))

In [ ]:
# fill
out[:30] = second_white_noise_ensemble_prime
out[30:40] = white_noise_ensemble_prime
out[40:] = S_mat_inference[np.newaxis, ...]

In [ ]:
# Build the ensemble

# stacked_matrices = np.vstack((out, S_mat_inference[np.newaxis, ...]))  # print(np.all(S_mat_inference == stacked_matrices[-1])) >> true

# S_mat_inference[np.newaxis, ...] creates a new axis because white_noise_ensemble_prime.shape = (10, 8191, 8191) so we need to make (1, 8191, 8191) matrix to stack properly

In [ ]:
# Average over the ensemble
ensemble_average = np.mean(out, axis=0)

In [ ]:
visualize_stress(ensemble_average, rows=f_inference, cols=t_dual_inference, smooth=True)

## Invert without smoothing

In [ ]:
def invert_wigner_function(S_mat, frequency_array, time_domain, xi_tilde_0=None):
    f = frequency_array
    t_vol = time_domain.scalar_dvol

    r_dom = time_domain
    h_dom = r_dom.get_default_codomain()
    FFT_forward = ift.FFTOperator(domain=(h_dom, r_dom), space=1) * (1/t_vol)

    S_mat_field = ift.Field(domain=ift.DomainTuple.make((h_dom, r_dom)), val=S_mat)

    Sigma_f_q = FFT_forward(S_mat_field).val

    # k: 1D frequency array length K
    # Sigma: 2D array shape (K, K) with axes (k1, k2)
    k_half = 0.5 * f               # desired first-axis locations
    # find nearest index in k for each k_half
    idx_rows = np.argmin(np.abs(f[:, None] - k_half[None, :]), axis=0)  # shape (K,)
    # pick one element per column j: Sigma[idx_rows[j], j]
    vec = Sigma_f_q[idx_rows, np.arange(len(f))]    # shape (K,)

    if xi_tilde_0 is not None:
        vec = vec / xi_tilde_0.conj()

    tmp = np.fft.ifft(vec, norm="ortho")

    return tmp

In [ ]:
reconstructed_xi = invert_wigner_function(S_mat=S_mat_inference, frequency_array=f_inference, time_domain=time_dom)

In [ ]:
plt.plot(t_dual_inference, reconstructed_xi)
usual_plot(title="Reconstructed xi field without smoothing of Wigner function")

## Implement some sort of smoothing

In [5]:
visualize_stress(S_mat_inference, rows=f_inference, cols=t_dual_inference, smooth=True)

		Rows must be in ascending order for visualization purposes but they are not, assuming a priori standard DFT order and moving DC to the middle


In [ ]:
from scipy.ndimage import gaussian_filter
sm = gaussian_filter(S_mat_inference, sigma=5.0)   # sigma ~ 0.5..3 blur radius in pixels

# threshhold for background subtraction

# mean_sm = np.mean(sm).real
# print("Smoothed mean value: ", mean_sm)
# thresh = mean_sm*10
# sm[np.where(sm.real<thresh)] = 0

# print("Applying threshhold: ", thresh)

In [ ]:
visualize_stress(sm, rows=f_inference, cols=t_dual_inference)

In [ ]:
reconstructed_xi = invert_wigner_function(S_mat=sm, frequency_array=f_inference, time_domain=time_dom)

In [ ]:
plt.plot(t_dual_inference, reconstructed_xi/max(reconstructed_xi)*max(nrt_strain_values), label="Reconstructed xi field (scaled)")
plt.plot(nrt_time_values, nrt_strain_values)
usual_plot(title="Reconstructed xi field after Gaussian smoothing of Wigner function")

## Shortcut: Invert only section of Wigner function

Periodic boundary conditions artifacts?

In [ ]:
t_i = 16.1085
t_f = 16.45
f_i = -350
f_f = -f_i

time_idcs_where_signal = np.where((t_dual_inference > t_i) & (t_dual_inference < t_f))[0]
freq_idcs_where_signal = np.where((f_inference > f_i) & (f_inference < f_f))[0]

print(time_idcs_where_signal.shape)  # these shapes must match right now for the inversion to work.. fix in the future
print(freq_idcs_where_signal.shape)

In [ ]:
times_where_signal = t_dual_inference[time_idcs_where_signal]
freqs_where_signal = f_inference[freq_idcs_where_signal]

In [ ]:
stress_where_signal = S_mat_inference[np.ix_(freq_idcs_where_signal,time_idcs_where_signal)]  # what is ix_???

In [ ]:
stress_where_signal_prime = stress_where_signal.copy()
stress_where_signal_prime[stress_where_signal<1e3]=0

In [ ]:
visualize_stress(stress_where_signal_prime, rows=freqs_where_signal, cols=times_where_signal, smooth=True)

In [ ]:
N_slice = len(times_where_signal)
dt_slice = times_where_signal[1]-times_where_signal[0]
time_dom_slice = ift.RGSpace(shape=(N_slice,), distances=dt_slice)

reconstructed_xi_from_slice = invert_wigner_function(S_mat=stress_where_signal_prime, frequency_array=freqs_where_signal, time_domain=time_dom_slice)

In [ ]:
plt.plot(times_where_signal, reconstructed_xi_from_slice/max(reconstructed_xi_from_slice)*max(nrt_strain_values), label="Reconstructed xi field (scaled)")
plt.plot(nrt_time_values, nrt_strain_values, label="Num. rel. template")
usual_plot(title="Reconstructed xi field after Gaussian smoothing and slicing Wigner function")

In [ ]:
ift.power_analyze()